# Tahap 1: Data Understanding & Exploratory Data Analysis (EDA)
## Dataset: Produk Pengondisi Udara (AC) - SIMEBTKE Kementerian ESDM Indonesia

**Sumber:** https://simebtke.esdm.go.id/sinergi/skem-label/konsumen/pengondisi-udara-ac  
**Tujuan:** Identifikasi pola efisiensi energi AC di Indonesia  

### Aturan Penelitian
1. Tidak mengubah data mentah
2. Salinan dataframe sebelum preprocessing
3. Tidak menghapus data tanpa alasan
4. Identifikasi missing value sebelum imputasi
5. Identifikasi kemungkinan data leakage
6. Tidak membuat kesimpulan sebelum melihat hasil
7. Tidak mengarang nilai atau pola
8. Semua transformasi dapat direproduksi
9. Visualisasi sesuai untuk publikasi ilmiah
10. Interpretasi statistik setelah analisis penting
11. Tandai setiap asumsi
12. Simpan preprocessing ke `data/processed`
13. Simpan grafik ke `outputs/figures`
14. Simpan tabel analisis ke `outputs/tables`

## 0. Setup & Konfigurasi

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 100,
    'savefig.dpi': 300,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'figure.facecolor': 'white',
})

for d in ['data/raw', 'data/processed', 'outputs/figures', 'outputs/tables']:
    os.makedirs(d, exist_ok=True)

RAW_PATH = 'data/raw/ac_simebtke_raw.csv'
print('Setup selesai.')

LoadError: ArgumentError: Package os not found in current path.
- Run `import Pkg; Pkg.add("os")` to install the os package.

## A. Load Dataset

**Aturan 1 & 2:** Tidak mengubah data mentah; buat salinan sebelum preprocessing.

**[ASEMSI]** Dataset dibaca dengan `dtype=str` untuk preservasi format mentah. Data asli di-fetch dari endpoint AJAX SIMEBTKE (lihat `fetch_data.py`).

In [ ]:
df_raw = pd.read_csv(RAW_PATH, dtype=str, encoding='utf-8-sig')
n_rows_raw, n_cols_raw = df_raw.shape
print(f'File: {RAW_PATH}')
print(f'Loaded {n_rows_raw} baris x {n_cols_raw} kolom')

df = df_raw.copy()

## B. Shape Dataset

In [ ]:
print(f'Jumlah baris : {df.shape[0]}')
print(f'Jumlah kolom : {df.shape[1]}')

## C. Nama dan Tipe Kolom

**[ASEMSI]** Semua kolom terbaca sebagai `str` karena `dtype=str` saat loading.

In [ ]:
print(df.dtypes.to_string())
print(f"\nTotal kolom object/string: {(df.dtypes == 'object').sum()}")

## D. 10 Baris Pertama

In [ ]:
df.head(10)

## E. Missing Values

Identifikasi missing value sebelum imputasi (aturan 4).

In [ ]:
for_missing = df.replace(['', 'null', 'NA', 'N/A', 'nan', 'None', '-'], np.nan)
missing_count = for_missing.isnull().sum()
missing_pct = (missing_count / len(for_missing) * 100).round(2)
missing_df = pd.DataFrame({
    'Jumlah Missing': missing_count,
    'Persentase (%)': missing_pct,
}).sort_values('Jumlah Missing', ascending=False)
missing_df = missing_df[missing_df['Jumlah Missing'] > 0]
missing_df

missing_full = pd.DataFrame({
    'Jumlah Missing': for_missing.isnull().sum(),
    'Persentase (%)': (for_missing.isnull().sum() / len(for_missing) * 100).round(2),
})
missing_full.to_csv('outputs/tables/01_missing_values.csv')
print('Tersimpan: outputs/tables/01_missing_values.csv')

## F. Duplicate Rows

In [ ]:
dup_all = df.duplicated().sum()
cols_no_no = [c for c in df.columns if c != 'NO.']
dup_no_no = df[cols_no_no].duplicated().sum()
reg = df['No. Registrasi/No. SHE'].replace('', np.nan).dropna()
dup_reg = reg.duplicated().sum()

print(f'Duplicate rows (semua kolom): {dup_all}')
print(f'Duplicate rows (tanpa NO.): {dup_no_no}')
print(f'Duplicate No. Registrasi/No. SHE: {dup_reg}')

if dup_reg > 0:
    dups = df[df['No. Registrasi/No. SHE'].isin(reg[reg.duplicated()])].sort_values('No. Registrasi/No. SHE')
    print(f'\nContoh duplikat No. Registrasi:')
    print(dups[['NO.', 'Merek', 'Model', 'No. Registrasi/No. SHE']].head(20).to_string())

pd.DataFrame({
    'Tipe Duplikat': ['Semua kolom', 'Tanpa NO.', 'No. Registrasi/No. SHE'],
    'Jumlah': [dup_all, dup_no_no, dup_reg],
}).to_csv('outputs/tables/02_duplicates.csv', index=False)

## G. Unique Values - Kolom Kategorikal

In [ ]:
categorical_cols = ['Merek', 'Famili', 'Model', 'Tipe', 'Rating Bintang (1-5)', 'LSPro']
cat_summary = []
for col in categorical_cols:
    n_unique = df[col].nunique()
    cat_summary.append({'Kolom': col, 'Jumlah Unique': n_unique})
    print(f'\n--- {col} ({n_unique} unique) ---')
    print(df[col].value_counts().head(10).to_string())

pd.DataFrame(cat_summary).to_csv('outputs/tables/03_categorical_unique.csv', index=False)

## H. Format Angka pada Kolom Numerik

Periksa apakah kolom numerik tersimpan sebagai string dengan karakter non-numerik (koma, 'Rp', 'W', 'BTU').

In [ ]:
numeric_cols = [
    'Daya (watt)',
    'Kapasitas Pendinginan (BTU/h)',
    'Nilai Efisiensi (EER/CSPF)',
    'Rating Bintang (1-5)',
    'Konsumsi Energi Tahunan (kWh)',
    'Biaya Listrik Tahunan (Rp)',
]

for col in numeric_cols:
    print(f'\n--- {col} ---')
    samples = df[col].dropna().unique()[:10]
    print(f'  Sample nilai mentah: {list(samples)}')
    has_comma = df[col].astype(str).str.contains(',', na=False).any()
    parsed_direct = pd.to_numeric(df[col], errors='coerce')
    n_parsed = parsed_direct.notna().sum()
    n_total = df[col].notna().sum()
    print(f'  Ada koma (,): {has_comma}')
    print(f'  Parse langsung berhasil: {n_parsed}/{n_total}')

print('\n[STRATEGI PARSING]')
print('Daya, Kapasitas, EER/CSPF, Rating, Konsumsi Energi: parse langsung')
print('Biaya Listrik: hapus koma thousand separator lalu parse')

## I. Format Tanggal

In [ ]:
date_cols = ['Tanggal Terbit SHE', 'SHE Berlaku Sampai Dengan Tanggal']
for col in date_cols:
    print(f'\n--- {col} ---')
    non_null = df[col].replace('', np.nan).dropna()
    print(f'  Non-null: {len(non_null)}/{len(df)}')
    if len(non_null) > 0:
        samples = non_null.unique()[:10]
        print(f'  Sample: {list(samples)}')
        parsed = pd.to_datetime(non_null, format='%Y-%m-%d', errors='coerce')
        n_parsed = parsed.notna().sum()
        print(f'  Parse ISO (YYYY-MM-DD): {n_parsed}/{len(non_null)} berhasil')
        if n_parsed < len(non_null):
            unparseable = non_null[parsed.isna()].unique()[:10]
            print(f'  Tidak terparse: {list(unparseable)}')

## J. Identifikasi Nilai Tidak Wajar

Parse numerik untuk inspeksi anomali (outlier, nilai 0, negatif, ekstrem).

In [ ]:
df_check = df.copy()
for col in ['Daya (watt)', 'Kapasitas Pendinginan (BTU/h)', 'Nilai Efisiensi (EER/CSPF)',
            'Konsumsi Energi Tahunan (kWh)']:
    df_check[col + '_num'] = pd.to_numeric(df_check[col], errors='coerce')
df_check['Biaya_num'] = pd.to_numeric(
    df_check['Biaya Listrik Tahunan (Rp)'].str.replace(',', '', regex=False), errors='coerce'
)
df_check['Rating_num'] = pd.to_numeric(df_check['Rating Bintang (1-5)'], errors='coerce')

for label, col, unit in [
    ('Daya (watt)', 'Daya (watt)_num', 'W'),
    ('Kapasitas Pendinginan (BTU/h)', 'Kapasitas Pendinginan (BTU/h)_num', 'BTU/h'),
    ('Nilai Efisiensi (EER/CSPF)', 'Nilai Efisiensi (EER/CSPF)_num', ''),
    ('Konsumsi Energi Tahunan (kWh)', 'Konsumsi Energi Tahunan (kWh)_num', 'kWh'),
    ('Biaya Listrik Tahunan (Rp)', 'Biaya_num', 'Rp'),
]:
    data = df_check[col].dropna()
    print(f'\n--- {label} ---')
    print(f'  Min: {data.min():.2f}, Max: {data.max():.2f}, Mean: {data.mean():.2f}')
    print(f'  Nilai 0: {(data == 0).sum()}, Nilai negatif: {(data < 0).sum()}')

In [ ]:
# Rating Bintang
rating = df_check['Rating_num'].dropna()
print(f'Rating Bintang: Min={rating.min()}, Max={rating.max()}')
print(f'Distribusi: {dict(rating.value_counts().sort_index())}')
print(f'Di luar rentang 1-5: {len(rating[(rating < 1) | (rating > 5)])}')

In [ ]:
# Daya tidak wajar
print('--- Daya < 100W ---')
print(df_check[df_check['Daya (watt)_num'] < 100][['NO.', 'Merek', 'Model', 'Daya (watt)']].to_string())
print('\n--- Daya > 5000W ---')
print(df_check[df_check['Daya (watt)_num'] > 5000][['NO.', 'Merek', 'Model', 'Daya (watt)']].head(10).to_string())

In [ ]:
# Cross-check: EER = Kapasitas / Daya
df_check['EER_calc'] = df_check['Kapasitas Pendinginan (BTU/h)_num'] / df_check['Daya (watt)_num']
df_check['EER_diff'] = abs(df_check['EER_calc'] - df_check['Nilai Efisiensi (EER/CSPF)_num'])
noninv = df_check[df_check['Tipe'] == 'Non-Inverter']
eer_diff_noninv = noninv['EER_diff'].dropna()
print(f'Non-Inverter: median selisih EER terhitung vs tercatat = {eer_diff_noninv.median():.4f}')
print(f'Non-Inverter: max selisih = {eer_diff_noninv.max():.4f}')
inv = df_check[df_check['Tipe'] == 'Inverter']
eer_diff_inv = inv['EER_diff'].dropna()
print(f'Inverter: median selisih CSPF terhitung vs tercatat = {eer_diff_inv.median():.4f}')
print('[ASEMSI] Untuk Inverter, nilai efisiensi adalah CSPF (bukan EER).')

# Cross-check: Biaya = Konsumsi * tarif
df_check['tarif_calc'] = df_check['Biaya_num'] / df_check['Konsumsi Energi Tahunan (kWh)_num']
tarif = df_check['tarif_calc'].dropna()
print(f'\nTarif terhitung: min={tarif.min():.2f}, median={tarif.median():.2f}, max={tarif.max():.2f}')
print('[ASEMSI] Tarif listrik PLN non-subsidi ~Rp 1,444/kWh (2023).')

## K. Ringkasan Statistik Numerik

In [ ]:
df_numeric = pd.DataFrame()
df_numeric['Daya (watt)'] = pd.to_numeric(df['Daya (watt)'], errors='coerce')
df_numeric['Kapasitas Pendinginan (BTU/h)'] = pd.to_numeric(df['Kapasitas Pendinginan (BTU/h)'], errors='coerce')
df_numeric['Nilai Efisiensi (EER/CSPF)'] = pd.to_numeric(df['Nilai Efisiensi (EER/CSPF)'], errors='coerce')
df_numeric['Rating Bintang'] = pd.to_numeric(df['Rating Bintang (1-5)'], errors='coerce')
df_numeric['Konsumsi Energi Tahunan (kWh)'] = pd.to_numeric(df['Konsumsi Energi Tahunan (kWh)'], errors='coerce')
df_numeric['Biaya Listrik Tahunan (Rp)'] = pd.to_numeric(
    df['Biaya Listrik Tahunan (Rp)'].str.replace(',', '', regex=False), errors='coerce'
)

desc = df_numeric.describe(include='all', percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc['count_nonnull'] = df_numeric.count()
desc['missing'] = df_numeric.isnull().sum()
desc['missing_pct'] = (df_numeric.isnull().sum() / len(df_numeric) * 100).round(2)
desc

desc.to_csv('outputs/tables/04_statistical_summary.csv')

In [ ]:
skew_kurt = pd.DataFrame({
    'Skewness': df_numeric.skew(numeric_only=True),
    'Kurtosis': df_numeric.kurtosis(numeric_only=True),
})
skew_kurt

skew_kurt.to_csv('outputs/tables/05_skewness_kurtosis.csv')

## L. Visualisasi Awal

### L1. Histogram Variabel Numerik

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Distribusi Variabel Numerik - Dataset AC SIMEBTKE', fontsize=14, fontweight='bold')

plot_cols = [
    ('Daya (watt)', 'Daya (watt)'),
    ('Kapasitas Pendinginan (BTU/h)', 'Kapasitas Pendingin (BTU/h)'),
    ('Nilai Efisiensi (EER/CSPF)', 'Nilai Efisiensi (EER/CSPF)'),
    ('Konsumsi Energi Tahunan (kWh)', 'Konsumsi Energi Tahunan (kWh)'),
    ('Biaya Listrik Tahunan (Rp)', 'Biaya Listrik Tahunan (Rp)'),
]
for idx, (col, label) in enumerate(plot_cols):
    ax = axes[idx // 3, idx % 3]
    data = df_numeric[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean={data.mean():.1f}')
    ax.axvline(data.median(), color='green', linestyle='--', linewidth=1.5, label=f'Median={data.median():.1f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=8)

ax = axes[1, 2]
rating_counts = df_numeric['Rating Bintang'].dropna().value_counts().sort_index()
bars = ax.bar(rating_counts.index, rating_counts.values, color='coral', edgecolor='white')
ax.set_xlabel('Rating Bintang')
ax.set_ylabel('Frekuensi')
for bar, val in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/figures/L1_histograms_numerik.png', bbox_inches='tight')
plt.show()

### L2. Boxplot per Tipe (Inverter vs Non-Inverter)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Boxplot per Tipe AC - Inverter vs Non-Inverter', fontsize=14, fontweight='bold')

df_plot = df_numeric.copy()
df_plot['Tipe'] = df['Tipe'].values

box_cols = [
    ('Daya (watt)', 'Daya (watt)'),
    ('Kapasitas Pendinginan (BTU/h)', 'Kapasitas (BTU/h)'),
    ('Nilai Efisiensi (EER/CSPF)', 'EER/CSPF'),
    ('Konsumsi Energi Tahunan (kWh)', 'Konsumsi (kWh)'),
    ('Biaya Listrik Tahunan (Rp)', 'Biaya (Rp)'),
    ('Rating Bintang', 'Rating Bintang'),
]
for idx, (col, label) in enumerate(box_cols):
    ax = axes[idx // 3, idx % 3]
    sns.boxplot(data=df_plot, x='Tipe', y=col, ax=ax, palette='Set2')
    ax.set_xlabel('')
    ax.set_ylabel(label)

plt.tight_layout()
plt.savefig('outputs/figures/L2_boxplot_per_tipe.png', bbox_inches='tight')
plt.show()

### L3. Top 20 Merek

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Top 20 Merek AC berdasarkan Jumlah Model Terdaftar', fontsize=14, fontweight='bold')

top_merek = df['Merek'].value_counts().head(20)
axes[0].barh(top_merek.index[::-1], top_merek.values[::-1], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Jumlah Model')
axes[0].set_title('Top 20 Merek (Semua)')

merek_tipe = df.groupby(['Merek', 'Tipe']).size().unstack(fill_value=0)
merek_tipe['Total'] = merek_tipe.sum(axis=1)
top_merek_tipe = merek_tipe.sort_values('Total', ascending=False).head(20).drop('Total', axis=1)
top_merek_tipe.plot(kind='barh', stacked=True, ax=axes[1], color=['steelblue', 'coral'])
axes[1].set_xlabel('Jumlah Model')
axes[1].set_title('Top 20 Merek (per Tipe)')
axes[1].legend(title='Tipe')

plt.tight_layout()
plt.savefig('outputs/figures/L3_top_merek.png', bbox_inches='tight')
plt.show()

### L4. Distribusi Rating Bintang per Tipe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Distribusi Rating Bintang Hemat Energi', fontsize=14, fontweight='bold')

rating_tipe = df_plot.groupby(['Rating Bintang', 'Tipe']).size().unstack(fill_value=0)
rating_tipe.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='white')
axes[0].set_xlabel('Rating Bintang')
axes[0].set_ylabel('Jumlah Model')
axes[0].set_title('Jumlah Model per Rating & Tipe')
axes[0].legend(title='Tipe')

rating_pct = rating_tipe.div(rating_tipe.sum(axis=0), axis=1) * 100
rating_pct.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], edgecolor='white')
axes[1].set_xlabel('Rating Bintang')
axes[1].set_ylabel('Persentase (%)')
axes[1].set_title('Proporsi Rating per Tipe')
axes[1].legend(title='Tipe')

plt.tight_layout()
plt.savefig('outputs/figures/L4_rating_per_tipe.png', bbox_inches='tight')
plt.show()

### L5. Scatter: Daya vs Kapasitas & Efisiensi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Hubungan Daya, Kapasitas, dan Efisiensi', fontsize=14, fontweight='bold')

scatter_df = df_numeric.copy()
scatter_df['Tipe'] = df['Tipe'].values
scatter_df['Merek'] = df['Merek'].values

sns.scatterplot(data=scatter_df, x='Daya (watt)', y='Kapasitas Pendinginan (BTU/h)',
                hue='Rating Bintang', style='Tipe', ax=axes[0], palette='RdYlGn', alpha=0.7, s=40)
axes[0].set_title('Daya vs Kapasitas Pendinginan')

sns.scatterplot(data=scatter_df, x='Daya (watt)', y='Nilai Efisiensi (EER/CSPF)',
                hue='Rating Bintang', style='Tipe', ax=axes[1], palette='RdYlGn', alpha=0.7, s=40)
axes[1].set_title('Daya vs Nilai Efisiensi (EER/CSPF)')

plt.tight_layout()
plt.savefig('outputs/figures/L5_scatter_daya_kapasitas_eer.png', bbox_inches='tight')
plt.show()

### L6. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Korelasi Pearson antar Variabel Numerik', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/L6_correlation_heatmap.png', bbox_inches='tight')
plt.show()

### L7. Konsumsi Energi & Biaya per Rating Bintang

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Efisiensi Energi & Biaya per Rating Bintang', fontsize=14, fontweight='bold')

sns.boxplot(data=df_plot, x='Rating Bintang', y='Konsumsi Energi Tahunan (kWh)', ax=axes[0], palette='RdYlGn')
axes[0].set_title('Konsumsi Energi per Rating Bintang')

sns.boxplot(data=df_plot, x='Rating Bintang', y='Biaya Listrik Tahunan (Rp)', ax=axes[1], palette='RdYlGn')
axes[1].set_title('Biaya Listrik per Rating Bintang')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('outputs/figures/L7_konsumsi_biaya_per_rating.png', bbox_inches='tight')
plt.show()

### L8. Distribusi LSPro

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
lspro_counts = df['LSPro'].replace('', np.nan).dropna().value_counts()
ax.barh(lspro_counts.index[::-1], lspro_counts.values[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Jumlah Model')
ax.set_title('Distribusi Lembaga Sertifikasi (LSPro)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/L8_lspro_distribution.png', bbox_inches='tight')
plt.show()

## M. Machine Learning

Tahap ini **TIDAK** melakukan machine learning. Fokus: data understanding dan EDA saja.

## Simpan Data yang Sudah Diparse (Reproducible)

Transformasi yang dapat direproduksi (aturan 8).

In [ ]:
df_processed = df.copy()

for col in ['Daya (watt)', 'Kapasitas Pendinginan (BTU/h)', 'Nilai Efisiensi (EER/CSPF)',
            'Rating Bintang (1-5)', 'Konsumsi Energi Tahunan (kWh)']:
    df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

df_processed['Biaya Listrik Tahunan (Rp)'] = pd.to_numeric(
    df_processed['Biaya Listrik Tahunan (Rp)'].str.replace(',', '', regex=False), errors='coerce'
)

for col in ['Tanggal Terbit SHE', 'SHE Berlaku Sampai Dengan Tanggal']:
    df_processed[col] = pd.to_datetime(df_processed[col], format='%Y-%m-%d', errors='coerce')

df_processed.to_csv('data/processed/ac_simebtke_parsed.csv', index=False, encoding='utf-8-sig')
print(f'Disimpan: data/processed/ac_simebtke_parsed.csv ({df_processed.shape[0]} x {df_processed.shape[1]})')
print(df_processed.dtypes.to_string())

## Laporan Ringkasan Tahap 1

### 1. Temuan Data Quality

| Aspek | Temuan |
|-------|--------|
| **Shape** | 1923 baris x 15 kolom (623 Inverter + 1300 Non-Inverter) |
| **Tipe data** | Semua kolom string saat loading; perlu parse numerik & tanggal |
| **Missing values** | Tanggal Terbit SHE: 75.82% missing; SHE Berlaku: 75.82%; LSPro: 57.88% |
| **Duplicates** | 0 duplikat penuh; 10 duplikat tanpa NO.; **541 duplikat No. Registrasi/No. SHE** |
| **Format angka** | 5 kolom numerik bersih; **Biaya Listrik** menggunakan koma thousand separator (hanya 27/1923 terparse langsung) |
| **Format tanggal** | ISO YYYY-MM-DD; 1 nilai `0000-00-00` tidak valid |
| **Nilai tidak wajar** | Daya min=1.16W (BEKO), max=20,400W (Midea); Konsumsi max=5,040,796 kWh; Biaya max=Rp 99,999,999.99; Biaya=0 untuk 27 baris |
| **Konsistensi** | Tarif listrik terhitung median=Rp 1,444.71/kWh (konsisten dengan PLN); EER cross-check: Non-Inverter median selisih=0.21 (wajar) |

### 2. Masalah yang Perlu Dibersihkan

1. **Biaya Listrik Tahunan (Rp)** — koma thousand separator harus dihapus sebelum parse numerik
2. **541 duplikat No. Registrasi/No. SHE** — perlu investigasi: apakah model berbeda dengan No. SHE yang sama (batch registration)?
3. **Nilai `0000-00-00` pada Tanggal Terbit SHE** — harus diperlakukan sebagai missing
4. **Outlier ekstrem** — Daya >5,000W (9 baris, mayoritas Midea multi-model), Konsumsi Energi >1,000,000 kWh, Biaya=Rp 99,999,999.99
5. **Missing 75.82% pada kolom tanggal SHE** — kemungkinan data lama tidak memiliki SHE digital
6. **Case inconsistency pada Merek** — `Gree` (135) vs `GREE` (79) kemungkinan mere yang sama
7. **Famili vs Model** — 1781 vs 1789 unique; banyak baris memiliki Famili=Model identik

### 3. Variabel yang Potensial Digunakan

| Variabel | Tipe | Potensi |
|----------|------|---------|
| Daya (watt) | Numerik | Prediktor utama efisiensi |
| Kapasitas Pendinginan (BTU/h) | Numerik | Prediktor utama; korelasi kuat dengan Daya |
| Nilai Efisiensi (EER/CSPF) | Numerik | **Target/variabel dependen** untuk analisis efisiensi |
| Rating Bintang (1-5) | Ordinal | Target klasifikasi; proxy efisiensi |
| Konsumsi Energi Tahunan (kWh) | Numerik | Outcome variabel; derived dari EER & Daya |
| Biaya Listrik Tahunan (Rp) | Numerik | Derived dari Konsumsi x tarif (~Rp 1,444/kWh) |
| Tipe (Inverter/Non-Inverter) | Kategorikal | Variabel grouping penting; EER vs CSPF |
| Merek | Kategorikal | 98 unique; perlu normalisasi case (Gree vs GREE) |
| LSPro | Kategorikal | 5 unique; 57.88% missing |

### 4. Potensi Data Leakage

1. **Biaya Listrik Tahunan = Konsumsi Energi x Tarif** — korelasi hampir perfect; **JANGAN** gunakan keduanya sebagai prediktor独立 dalam model yang memprediksi salah satunya
2. **Konsumsi Energi Tahunan** derived dari EER/CSPF & Daya — jika target adalah EER/CSPF, maka Konsumsi Energi adalah **leakage**
3. **Rating Bintang** derived dari EER/CSPF (threshold-based) — jika memprediksi Rating, jangan gunakan EER/CSPF sebagai prediktor
4. **Famili = Model** pada banyak baris — identik secara informasi; hanya gunakan salah satu

### 5. Analisis Lanjutan yang Disarankan

1. **Data Cleaning**: normalisasi Merek (case), hapus koma pada Biaya, tangani `0000-00-00`, investigasi 541 duplikat No. SHE
2. **Analisis outlier**: tentukan threshold wajar Daya (mis. <5,000W untuk AC residential) dan flag outlier
3. **Segmentasi**: analisis terpisah Inverter (CSPF) vs Non-Inverter (EER) karena metrik efisiensi berbeda
4. **Korelasi mendalam**: uji korelasi parsial Daya-Kapasitas-EER setelah kontrol Tipe
5. **Feature engineering**: rasio Kapasitas/Daya (EER terhitung), kategori PK, klasifikasi efisiensi
6. **Visualisasi lanjutan**: pairplot per Tipe, violin plot per Merek, trend waktu (Tanggal Terbit SHE)
7. **Statistical testing**: ANOVA/Kruskal-Wallis untuk perbedaan EER antar Rating Bintang dan Tipe
8. **Clustering (unsupervised)**: kelompokkan AC berdasarkan profil Daya-Kapasitas-EER
9. **Klasifikasi**: prediksi Rating Bintang dari Daya + Kapasitas + Tipe (tanpa leakage)
10. **Regresi**: prediksi EER/CSPF dari Daya + Kapasitas + Tipe (tanpa Konsumsi/Biaya)